# AEGAEON Lite — Kaggle

Import this notebook into Kaggle, add the distribution ZIP as an input, enable internet for downloads, and select an available GPU. The starter uses GPU 0 only, including when two GPUs are supplied.

This starter is code-reviewed and locally tested with simulated model responses. Live GPU/model quality is not yet measured. Never put valuable credentials in the runtime used to execute project code. Initial CUDA compilation can take several minutes; the model download is about 4.7 GB.


In [ ]:
from pathlib import Path

# Add aegaeon-v2.0.zip as a notebook input. Optionally add your task.zip too.
# Enable internet for runtime/model downloads and select a GPU accelerator.
DISTRIBUTION = next(Path("/kaggle/input").rglob("aegaeon-v2.0.zip"), None)
if DISTRIBUTION is None:
    raise FileNotFoundError("Add aegaeon-v2.0.zip as a notebook input first")
CUSTOM_TASK = next(Path("/kaggle/input").rglob("task.zip"), None)


In [ ]:
from pathlib import Path
import json, stat, sys, zipfile

PACKAGE_HOME = Path("/content/aegaeon-lite-app" if Path("/content").is_dir() else "/kaggle/working/aegaeon-lite-app")
with zipfile.ZipFile(DISTRIBUTION) as archive:
    entries = archive.infolist()
    if len(entries) > 200 or sum(e.file_size for e in entries) > 5_000_000:
        raise ValueError("Unexpected distribution size")
    seen = set()
    for entry in entries:
        parts = entry.filename.split("/")
        if (entry.is_dir() or entry.filename.startswith("/") or "\\" in entry.filename
            or ":" in entry.filename or any(p in {"", ".", ".."} for p in parts)
            or stat.S_ISLNK(entry.external_attr >> 16)
            or entry.filename.casefold() in seen):
            raise ValueError("Unsafe distribution entry")
        seen.add(entry.filename.casefold())
    for entry in entries:
        destination = PACKAGE_HOME / entry.filename
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(entry))
APP_ROOT = PACKAGE_HOME / "aegaeon-lite"
sys.path.insert(0, str(APP_ROOT))
from aegaeon_lite.notebook import setup_model, run_uploaded_task
print("AEGAEON Lite loaded from", APP_ROOT)


## Load the model

Pins llama.cpp and the official Qwen2.5-Coder-7B Q4_K_M weights, verifies the model checksum, checks hardware/disk, and starts loopback inference. If rerunning in the same session, stop the previous server first.


In [ ]:
# User-operated, loopback-only inference. No tunnel or remote worker.
RUNTIME_HOME = Path("/content/aegaeon-runtime" if Path("/content").is_dir() else "/kaggle/working/aegaeon-runtime")
RUNTIME = setup_model(RUNTIME_HOME, context=8192)
print(json.dumps(RUNTIME, indent=2))


## Validate the loaded model

Run the fixed JSON probe first. To test a real edit/check/review cycle, set RUN_MODEL_BENCHMARK=True below. The benchmark runs generated code; use a runtime without valuable credentials. Download the report even if it fails. A passed probe alone is not coding evidence.


In [ ]:
# Optional real-model smoke test. Runs only the generated calculator fixture.
# This executes model-written Python plus fixed regression tests in this runtime.
RUN_MODEL_BENCHMARK = False
from aegaeon_lite.provider import ModelProvider
from aegaeon_lite.validation import validate_model, save_report
from aegaeon_lite.notebook import gpu_info
from uuid import uuid4
VALIDATION_REPORT = RUNTIME_HOME / "validation-report.json"
gpu_before_validation = gpu_info()
validation = validate_model(
    ModelProvider(base_url=RUNTIME["endpoint"], model=RUNTIME["model"],
                  context_tokens=RUNTIME["context"]),
    benchmark_dir=(RUNTIME_HOME / ("validation-" + uuid4().hex[:8])) if RUN_MODEL_BENCHMARK else None)
validation["runtime"] = {key: RUNTIME[key] for key in (
    "llama_revision", "model_revision", "model_sha256", "context")}
validation["gpu_samples"] = {"before": gpu_before_validation, "after": gpu_info(),
    "notice": "Snapshots only, not peak VRAM or system RAM measurements."}
save_report(VALIDATION_REPORT, validation)
print(json.dumps(validation, indent=2))


In [ ]:
from IPython.display import FileLink, display
display(FileLink(str(VALIDATION_REPORT)))


## Choose and inspect a task

The included task fixes addition in a small calculator. For your own code, export task.zip using the PC CLI. Review check commands before enabling execution.


In [ ]:
from uuid import uuid4

# Use the included calculator task first, or select your uploaded task.zip.
TASK_BUNDLE = CUSTOM_TASK if CUSTOM_TASK is not None else APP_ROOT / "examples" / "calculator-task.zip"
# Only enable after inspecting manifest.json -> task -> checks. Commands execute
# code in this notebook runtime; this workspace is not an OS security sandbox.
ALLOW_CHECKS = False
# Set to a previously downloaded result.zip to restore after a runtime restart.
RESUME_RESULT = None
SESSION_HOME = RUNTIME_HOME / ("work-" + uuid4().hex[:8])
RESULT_ZIP = RUNTIME_HOME / "result.zip"
from aegaeon_lite.bundles import read_archive
task_manifest = json.loads(read_archive(TASK_BUNDLE)["manifest.json"])
print("Request:", task_manifest["task"]["request"])
print("Declared checks:", json.dumps(task_manifest["task"]["checks"], indent=2))


In [ ]:
# Set ALLOW_CHECKS=True in the preceding cell for the included trusted demo.
# Run this cell to inspect, edit, check and review one bounded task.
report = run_uploaded_task(TASK_BUNDLE, SESSION_HOME, RESULT_ZIP,
    allow_checks=ALLOW_CHECKS,
    provider=ModelProvider(base_url=RUNTIME["endpoint"], model=RUNTIME["model"], context_tokens=RUNTIME["context"]),
    resume_result=Path(RESUME_RESULT) if RESUME_RESULT else None)
print(json.dumps(report, indent=2))


## Download your checkpoint

Download after every task. On runtime loss, upload the original task ZIP and this result, set RESUME_RESULT to its path, then run in a fresh session. The PC importer previews the diff and can create a separate Git branch.


In [ ]:
from IPython.display import FileLink, display
# Download through this link or Kaggle's output file browser now.
display(FileLink(str(RESULT_ZIP)))
# Saving a notebook version/output is a separate platform action; do not assume
# temporary runtime files will survive session deletion.


## Stop inference when finished


In [ ]:
import os, signal
os.kill(RUNTIME["pid"], signal.SIGTERM)
